In [1]:
import os, sys

os.chdir(os.path.expanduser("~/QIAO0042/models/acv/facemask/"))
sys.path.insert(0, os.getcwd())
print("CWD:", os.getcwd())

CWD: /scratch-share/QIAO0042/models/acv/facemask


In [4]:
import torch
from torch.utils.data import DataLoader
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from unet import UNet
from palette import NUM_CLASSES
from dataset import FaceParsingDataset

# -----------------------------------------------------------------------
# Output palette: standard VOC-style, indices 0-15 are fixed, the rest
# fall back to greyscale (index == grey level).
# Pixel values in saved PNGs = class indices (0-18), NOT RGB colours.
# -----------------------------------------------------------------------
PALETTE = np.array([[i, i, i] for i in range(256)], dtype=np.uint8)
PALETTE[:16] = np.array([
    [0,   0,   0],    # 0  background
    [128, 0,   0],    # 1  skin
    [0,   128, 0],    # 2  nose
    [128, 128, 0],    # 3  eye_g
    [0,   0,   128],  # 4  l_eye
    [128, 0,   128],  # 5  r_eye
    [0,   128, 128],  # 6  l_brow
    [128, 128, 128],  # 7  r_brow
    [64,  0,   0],    # 8  l_ear
    [191, 0,   0],    # 9  r_ear
    [64,  128, 0],    # 10 mouth
    [191, 128, 0],    # 11 u_lip
    [64,  0,   128],  # 12 l_lip
    [191, 0,   128],  # 13 hair
    [64,  128, 128],  # 14 hat
    [191, 128, 128],  # 15 ear_r
], dtype=np.uint8)


def save_palette_png(label_hw: np.ndarray, path: str) -> None:
    """
    Save a (H, W) class-index array as a palette-mode PNG.
    Pixel value == class index (0-18).  The palette controls display colour only.
    """
    img = Image.fromarray(label_hw.astype(np.uint8), mode="P")
    img.putpalette(PALETTE.reshape(-1).tolist())
    img.save(path)

In [5]:
# ckpt_path = "checkpoints/full_best.pt"
ckpt_path = "checkpoints/unetv2_full.pt"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ckpt   = torch.load(ckpt_path, map_location=device, weights_only=False)

# Support both save formats:
#   train_with_full_dataset / train_split → ckpt["config"] with "base_width", "num_classes"
#   train_ddp.py                          → ckpt["args"]   with "base",       num_classes=NUM_CLASSES
if "config" in ckpt:
    cfg        = ckpt["config"]
    base_width = cfg["base_width"]
    n_classes  = cfg["num_classes"]
    dropout    = cfg.get("dropout", 0.0)
else:
    cfg        = ckpt["args"]
    base_width = cfg["base"]
    n_classes  = NUM_CLASSES
    dropout    = cfg.get("dropout", 0.0)

from unet import UNetV2
model = UNetV2(num_classes=n_classes, base=base_width, dropout=dropout).to(device)

# torch.compile prefixes all keys with "_orig_mod." — strip it before loading
def strip_compile_prefix(state_dict):
    return {k.replace("_orig_mod.", "", 1): v for k, v in state_dict.items()}

model.load_state_dict(strip_compile_prefix(ckpt["model"]))

# Load EMA shadow weights (smoothed weights → better predictions)
if "ema_shadow" in ckpt:
    ema_shadow = strip_compile_prefix(ckpt["ema_shadow"])
    for name, param in model.named_parameters():
        if name in ema_shadow:
            param.data.copy_(ema_shadow[name])
    print("Loaded EMA shadow weights")

model.eval()
print(f"Checkpoint : {ckpt_path}")
print(f"Epoch      : {ckpt['epoch']}")
print(f"Model      : UNetV2(base={base_width}, dropout={dropout})")
print(f"Device     : {device}")


Loaded EMA shadow weights
Checkpoint : checkpoints/unetv2_full.pt
Epoch      : 200
Model      : UNetV2(base=23, dropout=0.3)
Device     : cuda


In [ ]:
# --- TTA: average softmax(original) + softmax(flipped + label-swapped) ---
FLIP_PAIRS = [(4, 5), (6, 7), (8, 9)]  # l_eye↔r_eye, l_brow↔r_brow, l_ear↔r_ear

USE_TTA = False  # set True to enable test-time augmentation (hflip)

@torch.no_grad()
def predict(model, imgs, device):
    with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
        logits = model(imgs)

    if USE_TTA:
        with torch.amp.autocast("cuda", enabled=device.type == "cuda"):
            logits_flip = model(torch.flip(imgs, dims=[-1]))
        logits_flip = torch.flip(logits_flip, dims=[-1])
        logits_flip_swapped = logits_flip.clone()
        for a, b in FLIP_PAIRS:
            logits_flip_swapped[:, a] = logits_flip[:, b]
            logits_flip_swapped[:, b] = logits_flip[:, a]
        probs = (torch.softmax(logits, dim=1) +
                 torch.softmax(logits_flip_swapped, dim=1)) * 0.5
        return probs.argmax(dim=1)

    return logits.argmax(dim=1)


# --- run inference on test/images, save palette PNGs to test/predictions ---
out_dir = "test/predictions"
os.makedirs(out_dir, exist_ok=True)

val_ds     = FaceParsingDataset("test/images", mask_dir=None, augment=None)
val_loader = DataLoader(val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

saved = 0
with torch.no_grad():
    for imgs, fnames in val_loader:
        imgs = imgs.to(device, non_blocking=True)
        preds = predict(model, imgs, device).cpu().numpy()

        for pred, fname in zip(preds, fnames):
            out_path = os.path.join(out_dir, Path(fname).stem + ".png")
            save_palette_png(pred, out_path)
            saved += 1

print(f"Saved {saved} predictions  →  {out_dir}/  (TTA={'enabled' if USE_TTA else 'disabled'})")

# quick format check on one file
sample = Image.open(os.path.join(out_dir, Path(val_ds.img_paths[0].name).stem + ".png"))
arr    = np.array(sample)
print(f"Image mode  : {sample.mode}")
print(f"Pixel dtype : {arr.dtype}")
print(f"Value range : {arr.min()} – {arr.max()}")


In [ ]:
# --- visualise 8 random val samples (input image | palette-coloured prediction) ---
import random
random.seed(0)

n_show  = 8
indices = random.sample(range(len(val_ds)), n_show)
fig, axes = plt.subplots(n_show, 2, figsize=(10, n_show * 3))

for row, idx in enumerate(indices):
    img_tensor, fname = val_ds[idx]
    img_np    = img_tensor.permute(1, 2, 0).numpy()

    pred_path = os.path.join(out_dir, Path(fname).stem + ".png")
    # convert("RGB") applies the palette so matplotlib sees colours, not grey indices
    pred_rgb  = np.array(Image.open(pred_path).convert("RGB"))

    axes[row, 0].imshow(img_np)
    axes[row, 0].set_title(Path(fname).name[:30], fontsize=8)
    axes[row, 0].axis("off")

    axes[row, 1].imshow(pred_rgb)
    axes[row, 1].set_title("predicted mask (palette colours)")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.show()

In [7]:
import shutil
import zipfile

old_dir = out_dir
new_dir = "test/masks"

# Rename val/predictions -> val/masks (if needed)
if os.path.abspath(old_dir) != os.path.abspath(new_dir):
    if os.path.exists(new_dir):
        shutil.rmtree(new_dir)
    os.rename(old_dir, new_dir)

out_dir = new_dir  # keep notebook variable consistent

# Create val.zip with only val/masks/*
zip_path = "test.zip"
if os.path.exists(zip_path):
    os.remove(zip_path)

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for root, _, files in os.walk(out_dir):
        for f in files:
            full_path = os.path.join(root, f)
            zf.write(full_path, arcname=os.path.relpath(full_path, start="."))

print(f"Using masks dir: {out_dir}")
print(f"Created: {zip_path} (contains only val/masks)")

Using masks dir: test/masks
Created: test.zip (contains only val/masks)


In [8]:
if not os.path.exists(zip_path):
    print(f"Not found: {zip_path}")
else:
    with zipfile.ZipFile(zip_path, "r") as zf:
        members = sorted(zf.namelist())

    print(f"Archive: {zip_path}")
    print(f"Total entries: {len(members)}\n")

    # Build a simple tree
    tree = {}
    for path in members:
        parts = [p for p in path.split("/") if p]
        node = tree
        for part in parts:
            node = node.setdefault(part, {})

    def print_tree(node, prefix=""):
        keys = sorted(node.keys())
        for i, key in enumerate(keys):
            is_last = i == len(keys) - 1
            connector = "└── " if is_last else "├── "
            print(prefix + connector + key)
            extension = "    " if is_last else "│   "
            print_tree(node[key], prefix + extension)

    print_tree(tree)

Archive: test.zip
Total entries: 100

└── test
    └── masks
        ├── 02ca91df175b4db587759ddd25ca7957.png
        ├── 082cfddedd3141c98b60e4a501626722.png
        ├── 0ee42dbb79124fe1a73046a134e02bbf.png
        ├── 106c888e668e491aa74ff3078a5a6266.png
        ├── 1316dd1221f540128823a205a39324a3.png
        ├── 17684203dbc04dcbb91cce2f3e24ae5d.png
        ├── 18f63b3205924aa8956b816e493c7ca0.png
        ├── 1907cad3d8ed4025ac62a874e3674cb4.png
        ├── 1ca61dec3d25413e8ac95b7ef604b7c3.png
        ├── 1e9f8e51f5094272a8d60134606149ce.png
        ├── 20f5aa71187048f48a28a5ba34fbdf56.png
        ├── 214b1d88f27344f8a241f68681c3fee5.png
        ├── 21bf027ce0924a408d860306ffcc8116.png
        ├── 21fbdff0b064424d83599293496d4724.png
        ├── 2a4dd58ee09e469e8bd91e66f30f5f1c.png
        ├── 2af6646c3c364354838e31e2e3d72d25.png
        ├── 2b30dfdb538f41849fd2fadcff3ffc47.png
        ├── 2ca9a6ea303e4533a0c70d8e319582a8.png
        ├── 3045681a07fd49379bfbbed2a47f9145.png
        